Creating full Book Database:

In [2]:
# import function
%load_ext autoreload
%autoreload 2
from functions import read_file, out_csv, summarise_dataframe
import pandas as pd

yaml_path= "../config.yaml"
inp_data_section='raw_data'
file_name1= 'raw_scraped'    
file_name2= 'raw_api'    

#importing data file:
gr_df= read_file(yaml_path, inp_data_section, file_name1) #good reads books
ggl_df = read_file(yaml_path, inp_data_section, file_name2) #google books
print('========================')
summarise_dataframe(gr_df)
print('========================')
summarise_dataframe(ggl_df)

Shape: (675, 10)

--- Null counts ---
pages       1
pub_date    3
genres      1
dtype: int64

--- Dtypes ---
book_name          str
book_url           str
source_list        str
author             str
rating         float64
synopsis           str
pages          float64
pub_date           str
genres             str
img_url            str
dtype: object
Shape: (394, 11)

--- Null counts ---
author        4
rating      355
synopsis     57
pages        13
pub_date      4
genres       62
img_url      83
dtype: int64

--- Dtypes ---
book_name          str
book_url           str
source_list        str
author             str
rating         float64
synopsis           str
pages          float64
pub_date           str
genres             str
img_url            str
data_source        str
dtype: object


In [3]:

import re
from difflib import SequenceMatcher, get_close_matches

TITLE_CUTOFF  = 0.90
AUTHOR_CUTOFF = 0.85

def norm(s: str) -> str:
    """Lowercase, strip parentheticals/punctuation/leading articles."""
    if pd.isna(s):
        return ""
    s = str(s).lower()
    s = re.sub(r'\([^)]*\)', '', s)        # drop "(Harry Potter, #5)" etc.
    s = re.sub(r'[^a-z0-9\s]', ' ', s)     # drop punctuation
    s = re.sub(r'\s+', ' ', s).strip()
    for art in ('the ', 'a ', 'an '):
        if s.startswith(art):
            s = s[len(art):]
    return s

def sim(a: str, b: str) -> float:
    return SequenceMatcher(None, a, b).ratio()

# Normalize GR side once
gr_titles_norm  = gr_df['book_name'].map(norm).tolist()
gr_authors_norm = gr_df['author'].map(norm).tolist()
gr_ratings      = gr_df['rating'].tolist()

# Tag native Google Books ratings BEFORE overwriting
ggl_df['rating_source'] = ggl_df['rating'].notna().map({True: 'google_books', False: 'none'})

filled = 0
for i in ggl_df.index[ggl_df['rating'].isna()]:
    q_title  = norm(ggl_df.at[i, 'book_name'])
    q_author = norm(ggl_df.at[i, 'author'])
    if not q_title:
        continue
    # Fast title prefilter via stdlib
    candidates = get_close_matches(q_title, gr_titles_norm, n=5, cutoff=TITLE_CUTOFF)
    for cand in candidates:
        gr_idx = gr_titles_norm.index(cand)
        if sim(q_author, gr_authors_norm[gr_idx]) >= AUTHOR_CUTOFF:
            ggl_df.at[i, 'rating']        = gr_ratings[gr_idx]
            ggl_df.at[i, 'rating_source'] = 'goodreads_matched'
            filled += 1
            break

# Report
print(f"Ratings filled from GoodReads matches: {filled} / 355")
print(f"Still null after match:                {ggl_df['rating'].isna().sum()} / 394")
print("\nrating_source breakdown:")
print(ggl_df['rating_source'].value_counts())

Ratings filled from GoodReads matches: 25 / 355
Still null after match:                330 / 394

rating_source breakdown:
rating_source
none                 330
google_books          39
goodreads_matched     25
Name: count, dtype: int64


## Fill missing ratings from Open Library Search API

Google Books returned ~90% null ratings. For each null-rating row, query the Open Library Search API (`search.json`) on title + author and pull `ratings_average`. No API key required; free; rate-limited to 1 req/sec so runtime ≈ 6–7 min for ~355 books. Filled rows tagged `rating_source = 'open_library'`.

In [4]:

import time
import requests

OL_URL  = "https://openlibrary.org/search.json"
HEADERS = {"User-Agent": "IronhackW9-BookRec/1.0 (student project)"}
SLEEP   = 1.1   # respect 1 req/sec limit

def ol_rating(title: str, author: str) -> float | None:
    """Query Open Library, return ratings_average or None."""
    params = {
        "title":  str(title),
        "fields": "ratings_average,ratings_count",
        "limit":  1,
    }
    if pd.notna(author):
        params["author"] = str(author)
    try:
        r = requests.get(OL_URL, params=params, headers=HEADERS, timeout=15)
        r.raise_for_status()
        docs = r.json().get("docs", [])
        if docs and docs[0].get("ratings_average") is not None:
            return round(float(docs[0]["ratings_average"]), 2)
    except Exception as e:
        print(f"  ⚠️ {title[:40]!r}: {e}")
    return None

# Ensure rating_source exists (safe if Layer 1 already created it)
if "rating_source" not in ggl_df.columns:
    ggl_df["rating_source"] = ggl_df["rating"].notna().map(
        {True: "google_books", False: "none"}
    )

null_idx = ggl_df.index[ggl_df["rating"].isna()]
print(f"Querying Open Library for {len(null_idx)} books (~{len(null_idx)*SLEEP/60:.1f} min)...\n")

filled = 0
for n, i in enumerate(null_idx, 1):
    rating = ol_rating(ggl_df.at[i, "book_name"], ggl_df.at[i, "author"])
    if rating is not None:
        ggl_df.at[i, "rating"]        = rating
        ggl_df.at[i, "rating_source"] = "open_library"
        filled += 1
    if n % 25 == 0:
        print(f"  {n}/{len(null_idx)} checked, {filled} filled")
    time.sleep(SLEEP)

print(f"\n✅ Filled {filled} / {len(null_idx)} from Open Library")
print(f"Still null: {ggl_df['rating'].isna().sum()} / {len(ggl_df)}")
print("\nrating_source breakdown:")
print(ggl_df["rating_source"].value_counts())

Querying Open Library for 330 books (~6.1 min)...

  25/330 checked, 7 filled
  50/330 checked, 21 filled
  75/330 checked, 34 filled
  100/330 checked, 52 filled
  125/330 checked, 61 filled
  150/330 checked, 65 filled
  175/330 checked, 66 filled
  200/330 checked, 79 filled
  225/330 checked, 92 filled
  250/330 checked, 108 filled
  275/330 checked, 120 filled
  300/330 checked, 131 filled
  325/330 checked, 148 filled

✅ Filled 149 / 330 from Open Library
Still null: 181 / 394

rating_source breakdown:
rating_source
none                 181
open_library         149
google_books          39
goodreads_matched     25
Name: count, dtype: int64


In [32]:
print('========================')
summarise_dataframe(gr_df)
print('========================')
summarise_dataframe(ggl_df)

Shape: (675, 11)

--- Null counts ---
pages       1
pub_date    3
genres      1
dtype: int64

--- Dtypes ---
book_name          str
book_url           str
source_list        str
author             str
rating         float64
synopsis           str
pages          float64
pub_date           str
genres             str
img_url            str
data_source        str
dtype: object
Shape: (394, 11)

--- Null counts ---
author        4
rating      181
synopsis     57
pages        13
pub_date      4
genres       62
img_url      83
dtype: int64

--- Dtypes ---
book_name          str
book_url           str
source_list        str
author             str
rating         float64
synopsis           str
pages          float64
pub_date           str
genres             str
img_url            str
data_source        str
dtype: object


In [11]:
#getting dfs ready to concatenate
gr_df['data_source']='GoodReads'
ggl_df = ggl_df.drop(columns=["rating_source"])

KeyError: "['rating_source'] not found in axis"

In [12]:
print(gr_df.columns)
print(ggl_df.columns)

Index(['book_name', 'book_url', 'source_list', 'author', 'rating', 'synopsis',
       'pages', 'pub_date', 'genres', 'img_url', 'data_source'],
      dtype='str')
Index(['book_name', 'book_url', 'source_list', 'author', 'rating', 'synopsis',
       'pages', 'pub_date', 'genres', 'img_url', 'data_source'],
      dtype='str')


In [13]:
books_df = pd.concat([gr_df, ggl_df], ignore_index=True)

out_csv(
    df                  = books_df,
    yaml_path           = yaml_path,
    output_section_yaml = 'raw_data',
    file_name           = 'raw_combined'
)

print(f'Saved {len(books_df)} records.')

print(books_df.shape)            
print(books_df["data_source"].value_counts())
books_df.isna().sum()   # sanity check nulls 

(1069, 11)
data_source
GoodReads       675
google_books    394
Name: count, dtype: int64


book_name        0
book_url         0
source_list      0
author           4
rating         181
synopsis        57
pages           14
pub_date         7
genres          63
img_url         83
data_source      0
dtype: int64

In [19]:
books_df[books_df.genres.isnull()].source_list.unique()

<ArrowStringArray>
['Best Dystopian and Post-Apocalyptic Fiction',
                             'Best Books Ever',
                       'Books You Should Read',
                                 'Young Adult',
                 'Books That Should Be Movies',
                     '20th Century Literature',
                        'Best Book Boyfriends']
Length: 7, dtype: str

In [20]:
# 1. Define your mapping dictionary (List Name -> Genre)
genre_mapping = {
    'Best Dystopian and Post-Apocalyptic Fiction': 'Fiction|Dystopian',
    'Best Books Ever': 'general',
    'Books You Should Read': 'general',
    'Young Adult': 'general',
    'Books That Should Be Movies': 'adventure|fiction',
    '20th Century Literature': 'general',
    'Best Book Boyfriends': 'romance'
}

# 2. Fill missing values in 'genres' by mapping the 'source_list' column
books_df['genres'] = books_df['genres'].fillna(books_df['source_list'].map(genre_mapping))

In [26]:
books_df[books_df.author.isnull()]['book_name']

880        Five Go Adventuring Again
902       They Knew what They Wanted
1005        Moominvalley in November
1029    The Story of Doctor Dolittle
Name: book_name, dtype: str

In [27]:
# Fill in the missing authors by their index
books_df.loc[880, 'author'] = 'Enid Blyton'
books_df.loc[902, 'author'] = 'Sidney Howard'
books_df.loc[1005, 'author'] = 'Tove Jansson'
books_df.loc[1029, 'author'] = 'Hugh Lofting'

In [29]:
books_df[books_df.pub_date.isnull()]['book_name']

49                             The Odyssey
536       The Three Witches and The Master
581                         The Cheat Code
776    A Field Guide to the Addo Elephants
829           Hvor solen skinner om natten
833             Sådan opfinder man en pige
844                                Worbook
Name: book_name, dtype: str

In [30]:
# Filling missing 'pub_date' entries manually by index
books_df.loc[49, 'pub_date'] = '0001-01-01'
books_df.loc[536, 'pub_date'] = '2023-05-16'
books_df.loc[581, 'pub_date'] = '2023-05-26'
books_df.loc[776, 'pub_date'] = '2002-01-01'
books_df.loc[829, 'pub_date'] = '2018-01-16'
books_df.loc[833, 'pub_date'] = '2015-01-01'
books_df.loc[844, 'pub_date'] = '2003-01-01'

In [36]:
# Define the relative path to missing cover image
coverimg_path = 'figures/missingcover.png'
# Fill the missing values in the img_url column
books_df['img_url'] = books_df['img_url'].fillna(placeholder_path)

In [37]:
books_df.isna().sum()   # sanity check nulls 

book_name        0
book_url         0
source_list      0
author           0
rating         181
synopsis        57
pages           14
pub_date         0
genres           0
img_url          0
data_source      0
dtype: int64

In [1]:
books_df[books_df['book_name'].duplicated(keep=False)]

NameError: name 'books_df' is not defined

In [ ]:
#Ensure pub_date is in datetime format for accurate sorting
books_df['pub_date'] = pd.to_datetime(books_df['pub_date'], errors='coerce')

#Count how many non-null values each row has across all columns
# (Higher score = more complete information)
books_df['completeness_score'] = books_df.notna().sum(axis=1)

#Sort by completeness (highest first), then by pub_date (newest first)
# This guarantees that a complete row always beats an incomplete row.
books_df = books_df.sort_values(
    by=['completeness_score', 'pub_date'], 
    ascending=[False, False]
)

#Keep the best row for each book name
final_df = books_df.drop_duplicates(subset='book_name', keep='first')

#Clean up by dropping the temporary scoring column
final_df = final_df.drop(columns=['completeness_score'])

In [41]:
out_csv(
    df                  = final_df,
    yaml_path           = yaml_path,
    output_section_yaml = 'clean_data',
    file_name           = 'clean'
)

print(f'Saved {len(final_df)} records.')

File saved to: ../data/clean/booksDB.csv
Saved 1069 records.
